In [1]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [2]:
processor = AutoProcessor.from_pretrained("/home/ubuntu/Shree_FYP/data/stage1_unsloth")

The tokenizer you are loading from '/home/ubuntu/Shree_FYP/data/stage1_unsloth' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [3]:
model = AutoModelForImageTextToText.from_pretrained(
    "/home/ubuntu/Shree_FYP/data/stage1_unsloth",
    dtype=torch.bfloat16,
    device_map = "cuda")

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [2]:
processor = AutoProcessor.from_pretrained("unsloth/Qwen3.5-0.8B")

In [3]:
model = AutoModelForImageTextToText.from_pretrained(
    "unsloth/Qwen3.5-0.8B",
    dtype = torch.bfloat16,
    device_map = "cuda"
)

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [11]:
device = model.device
dummy_ids = torch.tensor([[1, 2, 3, 4, 5]], device=device)
dummy_mask = torch.ones_like(dummy_ids, device=device)

In [12]:
# 2. Run a single forward pass with cache enabled
with torch.no_grad():
    out = model(
        input_ids=dummy_ids,
        attention_mask=dummy_mask,
        use_cache=True,
        return_dict=True
    )

In [13]:
cache = out.past_key_values

In [14]:
print(f"\n🔍 Cache type: {type(cache)}")


🔍 Cache type: <class 'transformers.cache_utils.DynamicCache'>


In [16]:
from transformers.cache_utils import DynamicCache

In [17]:
# Safe attribute check
has_recurrent = hasattr(cache, 'recurrent_cache') or any('recurrent' in a.lower() for a in dir(cache))
has_kv       = hasattr(cache, 'key_cache') and hasattr(cache, 'value_cache')

In [18]:
if has_recurrent:
    print("   🟢 Recurrent state storage detected (GatedDeltaNet layers handled)")

   🟢 Recurrent state storage detected (GatedDeltaNet layers handled)


In [19]:
if has_kv:
    print("   🟢 Standard KV storage detected (GatedAttention layers handled)")

In [20]:
print(f"   📏 Sequence length tracked: {cache.get_seq_length()}")

   📏 Sequence length tracked: 5


In [25]:
# Inspect what DynamicCache actually built for your Qwen3.5 model
print(f"Cache type: {type(cache)}")
print(f"Number of cached layers: {len(cache.layers)}")

# Show layer types & state shapes
for i, layer in enumerate(cache.layers):
    if hasattr(layer, "keys") and layer.keys is not None:
        print(f"  Layer {i:2d}: keys shape={list(layer.keys.shape)} (growing KV cache)")
    elif hasattr(layer, "state") and layer.state is not None:
        print(f"  Layer {i:2d}: state shape={list(layer.state.shape)} (fixed recurrent state)")
    else:
        print(f"  Layer {i:2d}: empty/frozen (e.g., shared or non-cache layer)")

Cache type: <class 'transformers.cache_utils.DynamicCache'>
Number of cached layers: 24
  Layer  0: empty/frozen (e.g., shared or non-cache layer)
  Layer  1: empty/frozen (e.g., shared or non-cache layer)
  Layer  2: empty/frozen (e.g., shared or non-cache layer)
  Layer  3: keys shape=[1, 2, 5, 256] (growing KV cache)
  Layer  4: empty/frozen (e.g., shared or non-cache layer)
  Layer  5: empty/frozen (e.g., shared or non-cache layer)
  Layer  6: empty/frozen (e.g., shared or non-cache layer)
  Layer  7: keys shape=[1, 2, 5, 256] (growing KV cache)
  Layer  8: empty/frozen (e.g., shared or non-cache layer)
  Layer  9: empty/frozen (e.g., shared or non-cache layer)
  Layer 10: empty/frozen (e.g., shared or non-cache layer)
  Layer 11: keys shape=[1, 2, 5, 256] (growing KV cache)
  Layer 12: empty/frozen (e.g., shared or non-cache layer)
  Layer 13: empty/frozen (e.g., shared or non-cache layer)
  Layer 14: empty/frozen (e.g., shared or non-cache layer)
  Layer 15: keys shape=[1, 2, 5, 

In [26]:
# Qwen3.5 stores text config under .text_config or .config
cfg = getattr(model.config, "text_config", model.config)
layer_types = getattr(cfg, "layer_types", None)

if layer_types:
    print("Model layer_types config:")
    for i, lt in enumerate(layer_types[:12]):  # print first 12
        print(f"  Layer {i:2d}: {lt}")
else:
    print("layer_types not explicitly set; HF will default to full_attention")

Model layer_types config:
  Layer  0: linear_attention
  Layer  1: linear_attention
  Layer  2: linear_attention
  Layer  3: full_attention
  Layer  4: linear_attention
  Layer  5: linear_attention
  Layer  6: linear_attention
  Layer  7: full_attention
  Layer  8: linear_attention
  Layer  9: linear_attention
  Layer 10: linear_attention
  Layer 11: full_attention


In [27]:
from transformers.cache_utils import DynamicLayer, LinearAttentionLayer, DynamicSlidingWindowLayer

print("🔍 Cache Layer Type Inspection:")
for i, layer in enumerate(cache.layers):
    cls_name = type(layer).__name__
    
    # Identify behavior based on exact class
    if isinstance(layer, DynamicLayer):
        shape = list(layer.keys.shape) if layer.keys is not None else "empty"
        desc = f"Standard Attention KV Cache → {shape}"
    elif isinstance(layer, LinearAttentionLayer):
        # GatedDeltaNet / Mamba / Linear Attention
        state = getattr(layer, 'state', None)
        shape = list(state.shape) if state is not None else "fixed-size (initialized)"
        desc = f"Recurrent/Linear State → {shape}"
    elif isinstance(layer, DynamicSlidingWindowLayer):
        shape = list(layer.keys.shape) if layer.keys is not None else "empty"
        desc = f"Sliding Window KV → {shape}"
    else:
        desc = f"Unknown: {cls_name}"
        
    print(f"Layer {i:2d}: {cls_name:<35} | {desc}")

🔍 Cache Layer Type Inspection:
Layer  0: LinearAttentionLayer                | Recurrent/Linear State → fixed-size (initialized)
Layer  1: LinearAttentionLayer                | Recurrent/Linear State → fixed-size (initialized)
Layer  2: LinearAttentionLayer                | Recurrent/Linear State → fixed-size (initialized)
Layer  3: DynamicLayer                        | Standard Attention KV Cache → [1, 2, 5, 256]
Layer  4: LinearAttentionLayer                | Recurrent/Linear State → fixed-size (initialized)
Layer  5: LinearAttentionLayer                | Recurrent/Linear State → fixed-size (initialized)
Layer  6: LinearAttentionLayer                | Recurrent/Linear State → fixed-size (initialized)
Layer  7: DynamicLayer                        | Standard Attention KV Cache → [1, 2, 5, 256]
Layer  8: LinearAttentionLayer                | Recurrent/Linear State → fixed-size (initialized)
Layer  9: LinearAttentionLayer                | Recurrent/Linear State → fixed-size (initialized)

In [28]:
# Qwen3.5 stores text config under .text_config
cfg = getattr(model.config, "text_config", model.config)
layer_types = getattr(cfg, "layer_types", None)

if layer_types:
    print("\n📜 Config `layer_types` mapping:")
    for i, lt in enumerate(layer_types):
        print(f"  Layer {i:2d}: {lt}")
else:
    print("\n⚠️ `layer_types` not explicitly defined. HF infers from architecture defaults.")


📜 Config `layer_types` mapping:
  Layer  0: linear_attention
  Layer  1: linear_attention
  Layer  2: linear_attention
  Layer  3: full_attention
  Layer  4: linear_attention
  Layer  5: linear_attention
  Layer  6: linear_attention
  Layer  7: full_attention
  Layer  8: linear_attention
  Layer  9: linear_attention
  Layer 10: linear_attention
  Layer 11: full_attention
  Layer 12: linear_attention
  Layer 13: linear_attention
  Layer 14: linear_attention
  Layer 15: full_attention
  Layer 16: linear_attention
  Layer 17: linear_attention
  Layer 18: linear_attention
  Layer 19: full_attention
  Layer 20: linear_attention
  Layer 21: linear_attention
  Layer 22: linear_attention
  Layer 23: full_attention


In [9]:
# Run after loading your model
cfg = model.config.text_config
layer_types = getattr(cfg, "layer_types", None)

assert layer_types is not None, "❌ Missing `layer_types` in config. HF will default to full KV cache."
print("✅ Config `layer_types` found. First 12 layers:")
for i, lt in enumerate(layer_types[:12]):
    marker = "🟢 DeltaNet (Recurrent)" if "linear" in lt.lower() else "🔵 Attention (KV)"
    print(f"  Layer {i:2d}: {lt:<20} → {marker}")

✅ Config `layer_types` found. First 12 layers:
  Layer  0: linear_attention     → 🟢 DeltaNet (Recurrent)
  Layer  1: linear_attention     → 🟢 DeltaNet (Recurrent)
  Layer  2: linear_attention     → 🟢 DeltaNet (Recurrent)
  Layer  3: full_attention       → 🔵 Attention (KV)
  Layer  4: linear_attention     → 🟢 DeltaNet (Recurrent)
  Layer  5: linear_attention     → 🟢 DeltaNet (Recurrent)
  Layer  6: linear_attention     → 🟢 DeltaNet (Recurrent)
  Layer  7: full_attention       → 🔵 Attention (KV)
  Layer  8: linear_attention     → 🟢 DeltaNet (Recurrent)
  Layer  9: linear_attention     → 🟢 DeltaNet (Recurrent)
  Layer 10: linear_attention     → 🟢 DeltaNet (Recurrent)
  Layer 11: full_attention       → 🔵 Attention (KV)


In [16]:
import torch
from transformers.cache_utils import DynamicLayer, LinearAttentionLayer

device = model.device  # Your raw Qwen3_5ForConditionalGeneration instance
dummy_ids = torch.tensor([[1, 2, 3, 4, 5]], device=device)
dummy_mask = torch.ones_like(dummy_ids, device=device)

# 1. Run a standard forward pass to populate the cache
with torch.no_grad():
    outputs = model(
        input_ids=dummy_ids,
        attention_mask=dummy_mask,
        use_cache=True,
        return_dict=True
    )

cache = outputs.past_key_values  # This is now a DynamicCache object

# 2. Inspect the live cache layers
print(f"\n🔍 Cache class: {type(cache).__name__}")
print(f"🔍 Total cached layers: {len(cache.layers)}")

for i, layer in enumerate(cache.layers):
    if isinstance(layer, LinearAttentionLayer):
        print(f"  Layer {i:2d}: LinearAttentionLayer (DeltaNet recurrent state)")
    elif isinstance(layer, DynamicLayer):
        shape = list(layer.keys.shape) if layer.keys is not None else "empty"
        print(f"  Layer {i:2d}: DynamicLayer (Attention KV cache) → {shape}")
    else:
        print(f"  Layer {i:2d}: {type(layer).__name__} (custom/unused)")


🔍 Cache class: DynamicCache
🔍 Total cached layers: 32
  Layer  0: LinearAttentionLayer (DeltaNet recurrent state)
  Layer  1: LinearAttentionLayer (DeltaNet recurrent state)
  Layer  2: LinearAttentionLayer (DeltaNet recurrent state)
  Layer  3: DynamicLayer (Attention KV cache) → [1, 4, 5, 256]
  Layer  4: LinearAttentionLayer (DeltaNet recurrent state)
  Layer  5: LinearAttentionLayer (DeltaNet recurrent state)
  Layer  6: LinearAttentionLayer (DeltaNet recurrent state)
  Layer  7: DynamicLayer (Attention KV cache) → [1, 4, 5, 256]
  Layer  8: LinearAttentionLayer (DeltaNet recurrent state)
  Layer  9: LinearAttentionLayer (DeltaNet recurrent state)
  Layer 10: LinearAttentionLayer (DeltaNet recurrent state)
  Layer 11: DynamicLayer (Attention KV cache) → [1, 4, 5, 256]
  Layer 12: LinearAttentionLayer (DeltaNet recurrent state)
  Layer 13: LinearAttentionLayer (DeltaNet recurrent state)
  Layer 14: LinearAttentionLayer (DeltaNet recurrent state)
  Layer 15: DynamicLayer (Attention 

In [18]:
with torch.no_grad():
    # Step 1: prefix (seq_len=5)
    out1 = model(input_ids=dummy_ids, attention_mask=dummy_mask, use_cache=True)
    kv1 = out1.past_key_values.layers[3].keys
    print(f"Step 1 KV shape: {list(kv1.shape)}")  # [1, 4, 5, 256]

    # Step 2: generate one new token
    new_token = torch.tensor([[100]], device=device)  # dummy token id
    new_mask = torch.cat([dummy_mask, torch.ones(1,1,device=device)], dim=1)
    out2 = model(input_ids=new_token, attention_mask=new_mask, 
                 past_key_values=out1.past_key_values, use_cache=True)
    kv2 = out2.past_key_values.layers[3].keys
    print(f"Step 2 KV shape: {list(kv2.shape)}")  # [1, 4, 6, 256] ← seq_len grew by 1!

    # Step 3: one more
    new_token2 = torch.tensor([[101]], device=device)
    new_mask2 = torch.cat([new_mask, torch.ones(1,1,device=device)], dim=1)
    out3 = model(input_ids=new_token2, attention_mask=new_mask2,
                 past_key_values=out2.past_key_values, use_cache=True)
    kv3 = out3.past_key_values.layers[3].keys
    print(f"Step 3 KV shape: {list(kv3.shape)}")  # [1, 4, 7, 256] ← grew again!

Step 1 KV shape: [1, 4, 5, 256]
Step 2 KV shape: [1, 4, 6, 256]
Step 3 KV shape: [1, 4, 7, 256]


In [15]:
base = AutoModelForImageTextToText.from_pretrained(
    "unsloth/Qwen3.5-0.8B",
    dtype=torch.bfloat16,
    device_map = "cuda",
)

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [16]:
lora_cfg = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=[
        # Standard attention projections
        "q_proj", "k_proj", "v_proj", "o_proj",
        # Gated DeltaNet main projections
        "out_proj", "in_proj_qkv", "in_proj_z", "in_proj_b", "in_proj_a",
    ],
    bias="none",
)